In [5]:
"""
=============================================================================
AVALIAÇÃO DE MODELOS CLASSIFICADORES NO MNIST 
=============================================================================

Este notebook avalia 7 modelos de classificação usando validação cruzada,
ranqueia os modelos por acurácia e testa os top 3 no conjunto de teste, depois isso aplica o randomize para melhorar os melhores modelos.

Modelos avaliados:
1. Naive Bayes (GaussianNB)
2. MLP (Multi-layer Perceptron)
3. Random Forest (substituindo SVM por ser mais rápido)
4. Regressão Logística
5. SGD (Stochastic Gradient Descent)
6. KNN (K-Nearest Neighbors)
7. XGBoost
=============================================================================
"""
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import cross_val_predict, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Importando os modelos
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
import time
import warnings
warnings.filterwarnings('ignore')

print("✓ Bibliotecas importadas com sucesso!\n")



✓ Bibliotecas importadas com sucesso!



In [6]:
# 2. CARREGAMENTO E PREPARAÇÃO DOS DADOS

# Carregando o dataset MNIST
mnist = fetch_openml('mnist_784', version=1, parser='auto')
X, y = mnist.data, mnist.target

# Convertendo para arrays numpy e garantindo tipos corretos
X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)

# Dividindo em treino e teste (padrão MNIST: 60k treino, 10k teste)
X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

# Normalização dos dados (pixels de 0-255 para 0-1)
print("Normalizando dados (0-255 → 0-1)...")
X_train = X_train / 255.0
X_test = X_test / 255.0
print("✓ Normalização concluída!\n")

# 3. DEFINIÇÃO DOS MODELOS COM PARÂMETROS DEFAULT

# Dicionário com todos os modelos e seus parâmetros default
modelos = {
    'Naive Bayes': GaussianNB(),
    'MLP': MLPClassifier(random_state=42, max_iter=300),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Regressão Logística': LogisticRegression(random_state=42, max_iter=100),
    'SGD': SGDClassifier(random_state=42, max_iter=1000, tol=1e-3),
    'KNN': KNeighborsClassifier(),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='mlogloss')
}

# 4. VALIDAÇÃO CRUZADA E AVALIAÇÃO DOS MODELOS

# Dicionário para armazenar resultados
resultados = {}

#Loop para treinar e avaliar cada modelo
for nome_modelo, modelo in modelos.items():
    print("-" * 80)
    print(f"Avaliando: {nome_modelo}")
    print("-" * 80)
    
    tempo_inicio = time.time()
    
    try:
        # Realizando validação cruzada com 5 folds
        print(f"  → Executando cross_val_score (5-fold CV)...")
        cv_scores = cross_val_score(modelo, X_train, y_train, cv=5, 
                                     scoring='accuracy', n_jobs=-1, verbose=0)
        
        acuracia_media = cv_scores.mean()
        desvio_padrao = cv_scores.std()
        
        tempo_decorrido = time.time() - tempo_inicio
        
        # Armazenando resultados
        resultados[nome_modelo] = {
            'acuracia_media': acuracia_media,
            'desvio_padrao': desvio_padrao,
            'cv_scores': cv_scores,
            'tempo_segundos': tempo_decorrido
        }
        
        print(f"  ✓ Concluído!")
        print(f"    - Acurácia Média: {acuracia_media:.4f} (±{desvio_padrao:.4f})")
        print(f"    - Scores por fold: {cv_scores}")
        print(f"    - Tempo decorrido: {tempo_decorrido:.2f} segundos\n")
        
    except Exception as e:
        print(f"  ✗ Erro ao avaliar {nome_modelo}: {str(e)}\n")
        resultados[nome_modelo] = {
            'acuracia_media': 0.0,
            'desvio_padrao': 0.0,
            'cv_scores': None,
            'tempo_segundos': 0.0,
            'erro': str(e)
        }

Normalizando dados (0-255 → 0-1)...
✓ Normalização concluída!

--------------------------------------------------------------------------------
Avaliando: Naive Bayes
--------------------------------------------------------------------------------
  → Executando cross_val_score (5-fold CV)...
  ✓ Concluído!
    - Acurácia Média: 0.5618 (±0.0111)
    - Scores por fold: [0.55333333 0.57933333 0.55716667 0.54941667 0.56958333]
    - Tempo decorrido: 16.52 segundos

--------------------------------------------------------------------------------
Avaliando: MLP
--------------------------------------------------------------------------------
  → Executando cross_val_score (5-fold CV)...
  ✓ Concluído!
    - Acurácia Média: 0.9746 (±0.0014)
    - Scores por fold: [0.9765     0.975      0.97516667 0.97233333 0.97416667]
    - Tempo decorrido: 325.38 segundos

--------------------------------------------------------------------------------
Avaliando: Random Forest
------------------------------

## Ranking de Modelos Classificadores - MNIST Dataset
### Resultados da Validação Cruzada (5-Fold CV)

---

### 🏆 Ranking Completo (Ordem Decrescente de Acurácia)

| Posição | Modelo | Acurácia Média | Desvio Padrão | Tempo (s) |
| :---: | :--- | :---: | :---: | :---: |
| **1º** | **XGBoost** | **97.53%** | **±0.18%** | **656.68** |
| **2º** | **MLP** | **97.46%** | **±0.14%** | **325.38** |
| **3º** | **KNN** | **96.93%** | **±0.07%** | **77.46** |
| 4º | Random Forest | 96.64% | ±0.30% | 64.43 |
| 5º | Regressão Logística | 92.00% | ±0.43% | 43.38 |
| 6º | SGD | 90.70% | ±0.43% | 25.49 |
| 7º | Naive Bayes | 56.18% | ±1.11% | 16.52 |

In [7]:
# 5. AVALIAÇÃO DOS MODELOS NO CONJUNTO DE TESTE

# Dicionário para armazenar acurácias de teste
acuracias_teste = {}

for nome_modelo, modelo in modelos.items():
    print("-" * 80)
    print(f"Testando: {nome_modelo}")
    print("-" * 80)
    
    tempo_inicio = time.time()
    
    try:
        # Treinando o modelo com todo o conjunto de treino
        print(f"  → Treinando modelo")
        modelo.fit(X_train, y_train)
        
        # Fazendo predições no conjunto de teste
        print(f"  → Fazendo predições no teste")
        y_pred = modelo.predict(X_test)
        
        # Calculando acurácia no teste
        acuracia_teste = accuracy_score(y_test, y_pred)
        
        tempo_decorrido = time.time() - tempo_inicio
        
        acuracias_teste[nome_modelo] = acuracia_teste
        
        print(f"  ✓ Concluído!")
        print(f"    - Acurácia no Teste: {acuracia_teste:.4f}")
        print(f"    - Tempo decorrido: {tempo_decorrido:.2f} segundos\n")
        
    except Exception as e:
        print(f"  ✗ Erro ao testar {nome_modelo}: {str(e)}\n")
        acuracias_teste[nome_modelo] = 0.0

--------------------------------------------------------------------------------
Testando: Naive Bayes
--------------------------------------------------------------------------------
  → Treinando modelo
  → Fazendo predições no teste
  ✓ Concluído!
    - Acurácia no Teste: 0.5558
    - Tempo decorrido: 2.08 segundos

--------------------------------------------------------------------------------
Testando: MLP
--------------------------------------------------------------------------------
  → Treinando modelo
  → Fazendo predições no teste
  ✓ Concluído!
    - Acurácia no Teste: 0.9780
    - Tempo decorrido: 229.98 segundos

--------------------------------------------------------------------------------
Testando: Random Forest
--------------------------------------------------------------------------------
  → Treinando modelo
  → Fazendo predições no teste
  ✓ Concluído!
    - Acurácia no Teste: 0.9704
    - Tempo decorrido: 37.78 segundos

----------------------------------------

## Análise Comparativa: Validação Cruzada vs Conjunto de Teste
### Resultados dos Modelos com Hiperparâmetros Default

---

### TABELA 1: Comparação Completa CV vs Teste

| Ranking | Modelo | Acurácia CV (5-fold) | Acurácia Teste | Diferença (Teste - CV) | Variação (%) |
| :---: | :--- | :---: | :---: | :---: | :---: |
| **1º** | **XGBoost** | **97.53%** | **97.91%** | **+0.38%** | **+0.39%** |
| **2º** | **MLP** | **97.46%** | **97.80%** | **+0.34%** | **+0.35%** |
| 3º | Random Forest | 96.64% | 97.04% | +0.40% | +0.41% |
| 4º | KNN | 96.93% | 96.88% | -0.05% | -0.05% |
| 5º | Regressão Logística | 92.00% | 92.56% | +0.56% | +0.61% |
| 6º | SGD | 90.70% | 91.91% | +1.21% | +1.33% |
| 7º | Naive Bayes | 56.18% | 55.58% | -0.60% | -1.07% |

---

### TOP 3 Modelos - Análise Detalhada

🥇 **1º Lugar: XGBoost**
* **Acurácia CV:** 97.53% (±0.18%)
* **Acurácia Teste:** 97.91%
* **Diferença:** +0.38 pontos percentuais (+0.39%)
* **Status:** ✓ Excelente generalização - melhorou no teste!
* **Análise:** O melhor modelo geral. Performance consistente entre CV e teste, com leve melhora no conjunto de teste.

🥈 **2º Lugar: MLP (Multi-Layer Perceptron)**
* **Acurácia CV:** 97.46% (±0.14%)
* **Acurácia Teste:** 97.80%
* **Diferença:** +0.34 pontos percentuais (+0.35%)
* **Status:** ✓ Excelente generalização
* **Análise:** Performance muito próxima ao XGBoost. Também melhorou no teste, indicando boa capacidade de generalização.

🥉 **3º Lugar: KNN (K-Nearest Neighbors)**
* **Acurácia CV:** 96.93% (±0.07%)
* **Acurácia Teste:** 96.88%
* **Diferença:** -0.05 pontos percentuais (-0.05%)
* **Status:** ✓ Excelente generalização
* **Análise:** Mínima queda no teste (apenas 0.05%). O modelo mais estável (menor desvio padrão no CV).

---

### Insights sobre Generalização

**Modelos com Excelente Generalização (< 1% variação)**
* **XGBoost:** +0.39% → Melhorou no teste
* **MLP:** +0.35% → Melhorou no teste
* **KNN:** -0.05% → Praticamente igual
* **Random Forest:** +0.41% → Melhorou no teste

**Conclusão:** Os 4 melhores modelos apresentaram excelente capacidade de generalização, com variação menor que 0.5% entre CV e teste.

**Modelos com Atenção (> 1% variação)**
* **SGD:** +1.33% → Maior melhora no teste (possível underfitting no CV)
* **Naive Bayes:** -1.07% → Performance ruim em ambos os cenários

---

### 🔍 Análise Comportamental: CV vs Teste

**Padrão Observado: Melhora no Teste**
Dos 7 modelos, 5 melhoraram sua performance no conjunto de teste:
* XGBoost: +0.38%
* MLP: +0.34%
* Random Forest: +0.40%
* Regressão Logística: +0.56%
* SGD: +1.21%

**Interpretação:**
* ✓ Indica que os modelos não estão overfittados
* ✓ A validação cruzada foi conservadora (estimativa pessimista)
* ✓ O conjunto de teste pode ter sido ligeiramente mais fácil que alguns folds do CV

**Modelos que Pioraram (levemente)**
Apenas 2 modelos tiveram pequena queda:
* KNN: -0.05% (praticamente desprezível)
* Naive Bayes: -0.60% (modelo ruim em ambos cenários)

---

### Resposta à Pergunta: TOP 3 Escolhidos
Com base na performance no **conjunto de teste**, os TOP 3 são:

| Posição | Modelo | Acurácia Teste | Justificativa |
| :---: | :--- | :---: | :--- |
| **1º** | **XGBoost** | **97.91%** | Melhor acurácia absoluta + excelente generalização |
| **2º** | **MLP** | **97.80%** | Performance muito próxima ao XGBoost + rápido |
| **3º** | **Random Forest** | **97.04%** | Boa performance + melhorou significativamente no teste |

> **Nota:** O KNN (96.88%) foi preterido pelo Random Forest (97.04%) apesar de ter melhor CV, pois o objetivo é maximizar performance no teste.

---

**Por que esses 3?**
* ✅ Acurácia > 97% no teste
* ✅ Generalização comprovada (diferença CV-Teste < 0.5%)
* ✅ Diferentes paradigmas: Boosting (XGBoost), Neural Net (MLP), Bagging (Random Forest)

In [ ]:
#6. RANDOMIZED SEARCH - DEFINIÇÃO DOS HIPERPARÂMETROS")
from sklearn.model_selection import RandomizedSearchCV

print("=" * 80)

# Configuração geral (ajustada para velocidade/memória)
N_ITER = 8            # poucas combinações
N_CV = 2              # 2-fold CV (rápido)
RANDOM_STATE = 42
N_JOBS = 1            # evite usar -1 em máquinas com 8GB (muda para 2 se souber que tem memória)
VERBOSE = 1

print(f"CONFIG: n_iter={N_ITER}, cv={N_CV}, random_state={RANDOM_STATE}, n_jobs={N_JOBS}\n")

# -------------------------
# GRIDS ENXUTAS E RÁPIDAS
# -------------------------

param_dist_xgb = {
    'n_estimators': [50, 100, 150],        # poucas e pequenas
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.7, 0.9],
    'colsample_bytree': [0.7, 1.0],
    'min_child_weight': [1, 3],
    'gamma': [0, 0.1],
}

param_dist_mlp = {
    'hidden_layer_sizes': [(64,), (128,), (128, 64)],
    'activation': ['relu', 'tanh'],
    'alpha': [0.0001, 0.001],
    'learning_rate_init': [0.001, 0.01],
    'learning_rate': ['constant', 'adaptive'],
    'max_iter': [100, 200],    # iterações menores
    'batch_size': [32, 64],    # batches modestos
}

param_dist_rf = {
    'n_estimators': [50, 100, 150],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt'],  # fixado para reduzir ram e tempo
    'bootstrap': [True],
    'criterion': ['gini'],
}

# -------------------------
# FUNÇÃO AUXILIAR PARA RODAR O RANDOMIZEDSEARCH
# -------------------------
def run_random_search(model, param_dist, X, y, name):
    print("="*60)
    print(f"Iniciando RandomizedSearch: {name}")
    start = time.time()
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=N_ITER,
        cv=N_CV,
        scoring='accuracy',
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        verbose=VERBOSE,
    )
    search.fit(X, y)
    elapsed = time.time() - start
    print(f"-> {name} completo em {elapsed:.1f}s ({elapsed/60:.2f} min)")
    print(f"   best_score = {search.best_score_:.4f}")
    print(f"   best_params = {search.best_params_}\n")
    return {
        'name': name,
        'search': search,
        'best_score': search.best_score_,
        'best_params': search.best_params_,
        'time': elapsed
    }

# -------------------------
# EXECUÇÃO 
# -------------------------
resultados = {}
tempo_total_inicio = time.time()

# XGBoost
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss',
                          random_state=RANDOM_STATE, n_jobs=1)  # n_jobs=1 para XGB aqui
resultados['xgb'] = run_random_search(xgb_model, param_dist_xgb, X_train, y_train, "XGBoost")

# MLP
mlp_model = MLPClassifier(random_state=RANDOM_STATE, early_stopping=True)
resultados['mlp'] = run_random_search(mlp_model, param_dist_mlp, X_train, y_train, "MLP")

# Random Forest
rf_model = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1)
resultados['rf'] = run_random_search(rf_model, param_dist_rf, X_train, y_train, "RandomForest")

tempo_total = time.time() - tempo_total_inicio
print("="*60)
print(f"OTIMIZAÇÃO FINALIZADA em {tempo_total:.1f}s ({tempo_total/60:.2f} min)")

CONFIG: n_iter=8, cv=2, random_state=42, n_jobs=1

Iniciando RandomizedSearch: XGBoost
Fitting 2 folds for each of 8 candidates, totalling 16 fits
-> XGBoost completo em 7192.3s (119.87 min)
   best_score = 0.9684
   best_params = {'subsample': 0.7, 'n_estimators': 150, 'min_child_weight': 1, 'max_depth': 7, 'learning_rate': 0.2, 'gamma': 0.1, 'colsample_bytree': 1.0}

Iniciando RandomizedSearch: MLP
Fitting 2 folds for each of 8 candidates, totalling 16 fits
-> MLP completo em 5933.3s (98.89 min)
   best_score = 0.9729
   best_params = {'max_iter': 200, 'learning_rate_init': 0.001, 'learning_rate': 'constant', 'hidden_layer_sizes': (128, 64), 'batch_size': 32, 'alpha': 0.0001, 'activation': 'tanh'}

Iniciando RandomizedSearch: RandomForest
Fitting 2 folds for each of 8 candidates, totalling 16 fits
-> RandomForest completo em 349.2s (5.82 min)
   best_score = 0.9622
   best_params = {'n_estimators': 150, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth

In [13]:
"""
=============================================================================
 AVALIAÇÃO DOS BEST_MODELS EM NOVO CONJUNTO DE TESTE
=============================================================================

Este código:
1. Cria um NOVO split dos dados com semente diferente
2. Avalia cada best_model (do RandomizedSearch) no novo conjunto de teste
3. Compara os resultados: CV default → CV otimizado → Novo Teste

=============================================================================
"""

# =============================================================================
# 1. CRIANDO NOVO SPLIT COM SEMENTE DIFERENTE
# =============================================================================
from sklearn.model_selection import train_test_split
print("🔄 CRIANDO NOVO SPLIT DE DADOS")
print("-" * 80)

# Usando semente completamente diferente
RANDOM_STATE_NOVO = 999

print(f"Random state do novo split: {RANDOM_STATE_NOVO}")
print(f"(Diferente do split original e do RandomizedSearch que usou seed=42)\n")

# Criando novo split (mantendo os dados X, y que já foram carregados)
X_train_novo, X_test_novo, y_train_novo, y_test_novo = train_test_split(
    X, y, 
    test_size=0.15,  # 15% para teste (~10.5k amostras)
    random_state=RANDOM_STATE_NOVO, 
    stratify=y  # Mantém proporção das classes
)

print(f"✓ Novo split criado com sucesso!")
print(f"  Treino: {X_train_novo.shape[0]:,} amostras")
print(f"  Teste: {X_test_novo.shape[0]:,} amostras")
print(f"  Este conjunto de teste é TOTALMENTE DIFERENTE do anterior!\n")

# =============================================================================
# 2. EXTRAINDO OS BEST_MODELS DO RANDOMIZEDSEARCH
# =============================================================================
print("=" * 80)
print("📦 EXTRAINDO BEST_MODELS DO RANDOMIZEDSEARCH")
print("=" * 80)
print()

# Extraindo os melhores modelos de cada RandomizedSearch
best_models = {
    'XGBoost': resultados['xgb']['search'].best_estimator_,
    'MLP': resultados['mlp']['search'].best_estimator_,
    'Random Forest': resultados['rf']['search'].best_estimator_
}

print("✓ Best models extraídos:")
print(f"  • XGBoost (CV score: {resultados['xgb']['best_score']:.4f})")
print(f"  • MLP (CV score: {resultados['mlp']['best_score']:.4f})")
print(f"  • Random Forest (CV score: {resultados['rf']['best_score']:.4f})")
print()

# =============================================================================
# 3. AVALIANDO CADA BEST_MODEL NO NOVO CONJUNTO DE TESTE
# =============================================================================
print("=" * 80)
print("🧪 TESTANDO BEST_MODELS NO NOVO CONJUNTO DE TESTE")
print("=" * 80)
print()

resultados_novo_teste = {}

for nome_modelo, best_model in best_models.items():
    print("-" * 80)
    print(f"Avaliando: {nome_modelo}")
    print("-" * 80)
    
    tempo_inicio = time.time()
    
    # Fazendo predições no NOVO conjunto de teste
    y_pred_novo = best_model.predict(X_test_novo)
    acuracia_novo_teste = accuracy_score(y_test_novo, y_pred_novo)
    
    tempo_predicao = time.time() - tempo_inicio
    
    # Armazenando resultado
    resultados_novo_teste[nome_modelo] = acuracia_novo_teste
    
    print(f"  ✓ Acurácia no NOVO teste: {acuracia_novo_teste:.4f}")
    print(f"  Tempo de predição: {tempo_predicao:.2f}s\n")

🔄 CRIANDO NOVO SPLIT DE DADOS
--------------------------------------------------------------------------------
Random state do novo split: 999
(Diferente do split original e do RandomizedSearch que usou seed=42)

✓ Novo split criado com sucesso!
  Treino: 59,500 amostras
  Teste: 10,500 amostras
  Este conjunto de teste é TOTALMENTE DIFERENTE do anterior!

📦 EXTRAINDO BEST_MODELS DO RANDOMIZEDSEARCH

✓ Best models extraídos:
  • XGBoost (CV score: 0.9684)
  • MLP (CV score: 0.9729)
  • Random Forest (CV score: 0.9622)

🧪 TESTANDO BEST_MODELS NO NOVO CONJUNTO DE TESTE

--------------------------------------------------------------------------------
Avaliando: XGBoost
--------------------------------------------------------------------------------
  ✓ Acurácia no NOVO teste: 0.9791
  Tempo de predição: 0.51s

--------------------------------------------------------------------------------
Avaliando: MLP
--------------------------------------------------------------------------------
  ✓ 

## 🎯 TABELA PRINCIPAL: Evolução dos Modelos

| Modelo | CV Default | Teste 1 Default | CV Otimizado | Teste 2 Novo | Ganho CV | Ganho Teste |
|:-------|:----------:|:---------------:|:------------:|:------------:|:--------:|:-----------:|
| **XGBoost** | 97.53% | 97.91% | **96.84%** ⚠️ | 97.91% | **-0.69%** | 0.00% |
| **MLP** | 97.46% | 97.80% | **97.29%** | **98.59%** ✨ | **-0.17%** | **+0.79%** |
| **Random Forest** | 96.64% | 97.04% | **96.22%** ⚠️ | **98.60%** 🏆 | **-0.42%** | **+1.56%** |

---

## 🏆 Ranking Final - Teste 2 (Novo Split)

| Posição | Modelo | Acurácia | Tempo Predição | Performance |
|:-------:|:-------|:--------:|:--------------:|:------------|
| **🥇 1º** | **Random Forest** | **98.60%** | 1.14s | 🌟 **CAMPEÃO ABSOLUTO** |
| **🥈 2º** | **MLP** | **98.59%** | 0.35s ⚡ | Praticamente empatado, mais rápido |
| **🥉 3º** | **XGBoost** | **97.91%** | 0.51s | Ficou atrás, apesar de ser o melhor default |

**Diferença 1º vs 2º:** Apenas 0.01 ponto percentual! Tecnicamente **empate técnico**.

---

## 📈 Análise Detalhada 

### 🥇 Random Forest - O GRANDE VENCEDOR

| Métrica | Valor | Observação |
|:--------|:-----:|:-----------|
| **Acurácia Final** | **98.60%** | 🏆 Melhor resultado absoluto |
| **Ganho vs Default** | **+1.56%** | 🚀 Maior melhoria entre todos |
| **CV Otimizado** | 96.22% | ⚠️ Piorou no CV, mas... |
| **Generalização** | **Excelente** | Teste 2 superou expectativas! |
| **Tempo Predição** | 1.14s | Razoável |

**💡 Insight Chave:** O Random Forest teve uma **performance surpreendente** no Teste 2, apesar de ter piorado no CV durante a otimização. Isso sugere que:
- ✅ Os hiperparâmetros otimizados funcionaram melhor em dados novos
- ✅ O modelo generalizou excepcionalmente bem
- ✅ Possível underfitting benéfico (modelo menos complexo = melhor generalização)

In [14]:
import pickle

# Salvando o melhor modelo (Random Forest)
best_rf_model = resultados['rf']['search'].best_estimator_

# Exportando para arquivo pickle
with open('best_random_forest_mnist.pkl', 'wb') as f:
    pickle.dump(best_rf_model, f)

print("✓ Modelo Random Forest salvo como 'best_random_forest_mnist.pkl'")

# Salvando também os hiperparâmetros em um arquivo separado (opcional)
best_params = resultados['rf']['best_params']

with open('best_rf_params.pkl', 'wb') as f:
    pickle.dump(best_params, f)

print("✓ Hiperparâmetros salvos como 'best_rf_params.pkl'")

✓ Modelo Random Forest salvo como 'best_random_forest_mnist.pkl'
✓ Hiperparâmetros salvos como 'best_rf_params.pkl'
